# Playground Series S6E8 — Predicting Smartphone Addiction

Binary classification of `addicted_label` (0/1), scored with **ROC-AUC**. Submissions are probabilities,
so only the ranking matters — calibration is irrelevant here.

**Data:** 691,369 train / 296,302 test rows, 9 numerical + 3 categorical features.

Two properties drive everything that follows. First, **heavy missingness** — up to 19% in some columns,
with gaps in every column except `id`. Second, several numerical columns relate to the target
**non-monotonically**, which makes them look worthless to correlation and AUC while carrying a great deal
of information.

**Plan:** Overview → EDA → Preprocessing, constraint-aware imputation & target encoding →
10-fold CV with LightGBM / XGBoost / CatBoost → blend → submission.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from scipy.optimize import minimize
from scipy.stats import rankdata

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

SEED = 42
TARGET = 'addicted_label'

# 10 folds rather than 5: more training data per fold, and — more importantly — a richer
# lookup table for the exact-value target encoding introduced in section 3.
N_SPLITS = 10

# Competition metric is ROC-AUC (confirmed against the leaderboard: LB scores sit on the same
# scale as OOF AUC with a constant ~+0.0015 offset). Only the ranking matters, not calibration.
METRIC = 'auc'

In [ ]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../data/sample_submission.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('Sub shape  :', sample_sub.shape)

train_df.head()


## 1. Data Overview

In [ ]:
NUM_COLS = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time'
]
CAT_COLS = ['gender', 'stress_level', 'academic_work_impact']

print('--- dtypes ---')
print(train_df.dtypes)

print('\n--- describe (numeric) ---')
display(train_df[NUM_COLS + [TARGET]].describe().T.round(3))

print('--- categorical levels ---')
for c in CAT_COLS:
    print(f'{c}: {train_df[c].dropna().unique().tolist()}')

Every numerical column sits inside clean physical bounds — `age` 18–35, `daily_screen_time_hours` 0.5–15,
`sleep_hours` 4.5–9. There are no outliers or corrupted records, so no clipping is needed.
The categoricals are low-cardinality (2–3 levels), so either one-hot or label encoding works fine.

In [ ]:
# Missing value rate: train vs test
na_tbl = pd.DataFrame({
    'train_na_%': (train_df.drop(columns=[TARGET]).isna().mean() * 100).round(2),
    'test_na_%': (test_df.isna().mean() * 100).round(2),
}).sort_values('train_na_%', ascending=False)

display(na_tbl)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = na_tbl.drop(index='id').reset_index().melt(id_vars='index', var_name='split', value_name='pct')
sns.barplot(data=plot_df, y='index', x='pct', hue='split', palette='Set2', ax=ax)
ax.set_title('Missing Value Rate: Train vs Test')
ax.set_xlabel('% missing'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

Missingness is pervasive: `social_media_hours` 19.4%, `gaming_hours` 18.3%, `weekend_screen_time` 16.2%
— and **every column except `id`** has gaps. This is not data corruption; it looks deliberately injected
during dataset generation.

Note that the rates are not identical between train and test (`daily_screen_time_hours` is 13.9% in train
vs 11.1% in test). So the missingness pattern is a random mask drawn per split, which suggests missing
indicators will not carry signal — we test that below rather than assume it.

## 2. EDA

In [ ]:
# Target distribution
target_counts = train_df[TARGET].value_counts().sort_index()
target_pct = (target_counts / len(train_df) * 100).round(2)

print(pd.DataFrame({'count': target_counts, 'pct': target_pct}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette='Blues_d', ax=ax)
ax.set_title('Target Distribution (addicted_label)')
ax.set_xlabel('addicted_label'); ax.set_ylabel('count')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,.0f}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()

The positive class makes up **70.9%** of the data — mildly imbalanced, but nowhere near the point where
`scale_pos_weight` or resampling would help. The constant value in `sample_submission.csv` is exactly this
base rate (0.7094), i.e. the trivial baseline.

We still use **StratifiedKFold** so the class ratio stays stable across folds.

In [ ]:
# Numerical feature distributions, split by target
n_cols = 3
n_rows = int(np.ceil(len(NUM_COLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.6))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    sns.histplot(data=train_df, x=col, hue=TARGET, bins=50, ax=axes[i],
                 palette='Set2', stat='density', common_norm=False, element='step')
    axes[i].set_title(col); axes[i].set_xlabel('')

for j in range(len(NUM_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions by Target', fontsize=14, y=1.005)
plt.tight_layout(); plt.show()

`daily_screen_time_hours`, `weekend_screen_time` and `social_media_hours` visibly separate the two classes
— the positive-class distributions are clearly shifted right. In contrast `age`, `sleep_hours`,
`notifications_per_day` and `app_opens_per_day` almost completely overlap, so they carry little
information on their own.

In [ ]:
# Target rate per feature decile -> shows monotonicity and signal strength
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.flatten()
base_rate = train_df[TARGET].mean()

single_auc = {}
for i, col in enumerate(NUM_COLS):
    tmp = train_df[[col, TARGET]].dropna()
    single_auc[col] = roc_auc_score(tmp[TARGET], tmp[col])

    bins = pd.qcut(tmp[col], 10, duplicates='drop')
    rate = tmp.groupby(bins, observed=True)[TARGET].mean()
    axes[i].plot(range(len(rate)), rate.values, marker='o', color='#2b6cb0')
    axes[i].axhline(base_rate, ls='--', c='grey', lw=1)
    axes[i].set_title(f'{col}\nsingle-feature AUC = {single_auc[col]:.3f}', fontsize=10)
    axes[i].set_xlabel('decile'); axes[i].set_ylabel('P(addicted)')
    axes[i].set_ylim(0, 1.05)

plt.suptitle('Target Rate by Feature Decile', fontsize=14, y=1.005)
plt.tight_layout(); plt.show()

print(pd.Series(single_auc).sort_values(ascending=False).round(4))

This is the most informative plot in the EDA. Three observations:

1. **The screen-time features are nearly deterministic.** `daily_screen_time_hours` reaches P(addicted) ≈ 1.00
   in the top decile and ≈ 0.26 in the bottom one (single-feature AUC 0.889). `weekend_screen_time` (0.880)
   and `social_media_hours` (0.858) behave the same way. The target is largely a function of these three.
2. **Those three relationships are monotonic and smooth** — no sharp thresholds, just a sigmoid-like
   transition. That points to a label generated from a weighted score plus noise, which tree models
   capture easily.
3. **`age` (0.503), `notifications_per_day` (0.493), `sleep_hours` (0.529) and `app_opens_per_day` (0.541)
   score at chance — but do not read that as "no signal".** Look at their curves rather than their AUC:
   `app_opens_per_day` swings 0.645 → 0.770 across deciles of ~61,000 rows, about 65 standard errors.
   The relationship is strong and simply **not monotonic**, which is exactly what AUC cannot see.

Point 3 is the thread that section 3 picks up with target encoding, and it turns out to be the single
largest source of improvement in this notebook — `notifications_per_day` goes from 0.492 to 0.749 once it
is encoded by exact value.

In [ ]:
# Categorical features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(CAT_COLS):
    g = train_df.groupby(col)[TARGET].agg(['mean', 'count'])
    sns.barplot(x=g.index, y=g['mean'], palette='Set2', ax=axes[i])
    axes[i].axhline(base_rate, ls='--', c='grey', lw=1)
    axes[i].set_title(col); axes[i].set_ylabel('P(addicted)')
    axes[i].set_ylim(0.6, 0.8)
    for p, v in zip(axes[i].patches, g['mean']):
        axes[i].annotate(f'{v:.3f}', (p.get_x() + p.get_width() / 2, v), ha='center', va='bottom', fontsize=9)

plt.suptitle('Target Rate by Categorical Feature (dashed = base rate 0.709)', fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

for col in CAT_COLS:
    print(train_df.groupby(col)[TARGET].agg(['mean', 'count']).round(4), '\n')

All three categoricals are essentially signal-free. The largest deviation is `gender=Male` at 0.723 versus
the 0.709 base rate — about 1.4 points. The levels of `stress_level` and `academic_work_impact` differ by
less than half a point.

We keep them in the model (they may still contribute through interactions), but there is no point building
target encoding on top of these columns: no gain to capture, and it would only add leakage risk.

In [ ]:
# Correlation matrix
corr = train_df[NUM_COLS + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout(); plt.show()

The only strong relationship between features is `daily_screen_time_hours` ↔ `weekend_screen_time`
(r = 0.80), which is expected since both measure the same behaviour. Every other pair sits below |r| < 0.5.
There is no multicollinearity problem worth dropping columns over.

In [ ]:
# Is missingness related to the target? (MCAR check)
rows = []
for col in NUM_COLS + CAT_COLS:
    m = train_df[col].isna()
    rows.append({
        'feature': col,
        'na_rate': round(m.mean(), 4),
        'P(y=1 | NA)': round(train_df.loc[m, TARGET].mean(), 4),
        'P(y=1 | not NA)': round(train_df.loc[~m, TARGET].mean(), 4),
    })
na_signal = pd.DataFrame(rows)
na_signal['diff'] = (na_signal['P(y=1 | NA)'] - na_signal['P(y=1 | not NA)']).round(4)
display(na_signal.sort_values('diff', key=abs, ascending=False))

The largest gap between "missing" and "not missing" is 0.004 against a 0.709 base rate — statistically
noise. The data is **MCAR**: values were deleted at random, independently of the target.

The practical consequence is that `*_is_na` indicator features carry no signal by themselves. We still keep
a single `n_missing` count, since knowing *how many* fields are unknown can help the model calibrate its
uncertainty, and let CV decide whether it earns its place.

In [ ]:
# Structural constraint check between features (on complete rows)
cc = train_df.dropna(subset=NUM_COLS)
print(f'Complete rows: {len(cc):,} ({len(cc)/len(train_df):.1%})\n')

check1 = (cc['daily_screen_time_hours'] >= cc['social_media_hours'] + cc['gaming_hours'] + cc['work_study_hours']).mean()
check2 = (cc['daily_screen_time_hours'] + cc['sleep_hours'] <= 24).mean()
print(f'daily_screen >= social + gaming + work_study : {check1:.4%} of rows')
print(f'daily_screen + sleep <= 24                   : {check2:.4%} of rows')

residual = (cc['daily_screen_time_hours'] - cc['social_media_hours']
            - cc['gaming_hours'] - cc['work_study_hours'])
print('\nresidual screen time (daily - social - gaming - work_study):')
print(residual.describe().round(3))

This check reveals an important structure: in **100% of rows**, `daily_screen_time_hours` ≥
`social_media_hours + gaming_hours + work_study_hours`. Daily screen time is therefore the sum of these
three activities plus an uncategorised residual.

That residual ("other screen time") is a meaningful feature: it appears in no raw column, yet it measures
how much of a person's screen time is unaccounted for. We derive it below, along with related share and
ratio features.

This identity is also the basis for the constraint-aware imputation in the next section — it turns
missing values from unknowns into *bounded* unknowns.

## 3. Preprocessing & Feature Engineering

The decisions and the reasoning behind them:

**1) We do not impute missing values.** LightGBM, XGBoost and CatBoost all handle missingness natively —
at every split they send NaNs down whichever branch maximises gain. Median/mean imputation, at a 19%
missing rate, would artificially sharpen the distribution and hide the "this value is unknown" signal from
the model. The one exception is CatBoost, which rejects NaN in categorical columns, so there we encode
missingness as an explicit `'Missing'` level — that is categorical encoding, not imputation.

**2) The derived features come from two ideas:**
- *Composition:* Starting from the structural constraint above, the components of screen time
  (`other_screen`, `social_share`, `gaming_share`, `work_share`).
- *Density / normalisation:* Relative measures instead of absolute hours — screen share of waking time
  (`screen_share_awake`), notifications per app open (`notif_per_open`), minutes per open
  (`minutes_per_open`), and weekend-versus-weekday behaviour (`weekend_gap`, `weekend_ratio`).
  These describe behavioural patterns rather than raw scale.

**3) Categoricals stay model-native:** `category` dtype for LightGBM, `enable_categorical=True` for
XGBoost, `cat_features` for CatBoost. With cardinality of 2–3 there is no need for target encoding, and the
EDA already showed these columns carry no signal — TE would only add leakage risk.

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    D  = df['daily_screen_time_hours']
    S  = df['social_media_hours']
    G  = df['gaming_hours']
    W  = df['work_study_hours']
    SL = df['sleep_hours']
    N  = df['notifications_per_day']
    O  = df['app_opens_per_day']
    WK = df['weekend_screen_time']

    # --- composition: the components of screen time ---
    df['other_screen']   = D - S - G - W          # uncategorised screen time
    df['leisure_screen'] = D - W                  # screen time outside work/study
    df['social_gaming']  = S + G                  # entertainment-driven usage
    df['social_share']   = S / D
    df['gaming_share']   = G / D
    df['work_share']     = W / D

    # --- normalisation: relative rather than absolute measures ---
    df['awake_hours']        = 24 - SL
    df['screen_share_awake'] = D / (24 - SL)      # share of waking hours spent on screen
    df['screen_sleep_ratio'] = D / SL

    # --- weekly rhythm ---
    df['week_avg_screen'] = (5 * D + 2 * WK) / 7
    df['weekend_gap']     = WK - D
    df['weekend_ratio']   = WK / D

    # --- notification / interaction intensity ---
    df['notif_per_open']    = N / O
    df['minutes_per_open']  = D * 60 / O
    df['notif_per_hour']    = N / D
    df['opens_per_hour']    = O / D

    # --- missingness density ---
    df['n_missing'] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1).astype('int8')

    return df


ENG_COLS = [
    'other_screen', 'leisure_screen', 'social_gaming', 'social_share', 'gaming_share', 'work_share',
    'awake_hours', 'screen_share_awake', 'screen_sleep_ratio',
    'week_avg_screen', 'weekend_gap', 'weekend_ratio',
    'notif_per_open', 'minutes_per_open', 'notif_per_hour', 'opens_per_hour',
    'n_missing',
]

train_fe = add_features(train_df)
test_fe  = add_features(test_df)

FEATURES = NUM_COLS + CAT_COLS + ENG_COLS
print(f'{len(FEATURES)} features ({len(NUM_COLS)} num + {len(CAT_COLS)} cat + {len(ENG_COLS)} engineered)')
train_fe[ENG_COLS].describe().T.round(3)

### Constraint-aware imputation

Section 2 verified a hard identity that holds on **100% of complete rows**:

```
daily_screen_time = social_media + gaming + work_study + other,   other >= 0
```

This is stronger than a correlation — it is an arithmetic constraint, and it turns missing values from
unknowns into **bounded** unknowns:

- `daily` missing, parts known → `daily >= social + gaming + work`, with `daily ≈ that sum + median(other)`
- `daily` known, one part missing → that part `<= daily - (the other two)`

Gradient boosted trees cannot recover this on their own. They split one feature at a time and never form a
sum across columns, so `S + G + W` is invisible to them no matter how deep the trees grow. Supplying the
bound explicitly is genuinely new information rather than a restatement of what the model already sees.

This matters because `daily_screen_time_hours` is the strongest single feature in the dataset
(AUC 0.889) and it is missing in 13.9% of train / 11.1% of test rows. Recovering even part of it should
show up directly in the score.

The cell below first prints the coverage — what fraction of rows the constraint can actually be applied to.
That number is the ceiling on what this can buy, so it is worth reading before trusting the idea.

Fallback chain for `daily`: constraint estimate first, then the weekend relationship
(`weekend_screen_time` correlates 0.80 with daily) for rows where no part is known. All the constants
(median residual, component shares, weekend ratio) are fitted on train only, and describe feature
distributions rather than the target — so there is no leakage to guard against.

In [ ]:
# --- Constraint-aware imputation ---
D_, S_, G_, W_, SL_, WK_ = ('daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                            'work_study_hours', 'sleep_hours', 'weekend_screen_time')

# Coverage check: how often is the constraint actually usable? This bounds the possible gain.
for nm, df_ in (('train', train_fe), ('test', test_fe)):
    d_na = df_[D_].isna()
    parts_known = df_[[S_, G_, W_]].notna().sum(axis=1)
    print(f'--- {nm} ---')
    print(f'  daily missing, all of S/G/W known : {(d_na & (parts_known == 3)).mean():.2%}')
    print(f'  daily missing, at least one known : {(d_na & (parts_known >= 1)).mean():.2%}')
    print(f'  daily known, some part missing    : {(~d_na & (parts_known < 3)).mean():.2%}')

# Statistics are fitted on TRAIN only. They describe feature distributions and never touch the
# target, so there is no leakage and no need to make them fold-safe.
_cc = train_fe.dropna(subset=[D_, S_, G_, W_])

# median share each component takes of the room available to it
OTHER_FRAC = {}
for c in (S_, G_, W_):
    others = [x for x in (S_, G_, W_) if x != c]
    room = _cc[D_] - _cc[others].sum(axis=1)
    OTHER_FRAC[c] = float((_cc[c] / room.clip(lower=1e-6)).median())

# median slack (D - known parts), conditioned on how many parts are known
_d_known = train_fe[train_fe[D_].notna()]
SLACK_MED = (_d_known[D_] - _d_known[[S_, G_, W_]].fillna(0).sum(axis=1)) \
    .groupby(_d_known[[S_, G_, W_]].notna().sum(axis=1)).median()

_wk = train_fe.dropna(subset=[D_, WK_])
WK_RATIO = float((_wk[WK_] / _wk[D_]).median())

print(f'\ncomponent share of room: { {k: round(v, 3) for k, v in OTHER_FRAC.items()} }')
print('median slack by #known parts:', SLACK_MED.round(3).to_dict())
print('median weekend/daily ratio  :', round(WK_RATIO, 3))


def add_constraint_features(df):
    df = df.copy()
    D, S, G, W, SL, WK = (df[D_], df[S_], df[G_], df[W_], df[SL_], df[WK_])

    parts = df[[S_, G_, W_]]
    n_known = parts.notna().sum(axis=1)
    partial = parts.fillna(0).sum(axis=1)

    df['sgw_n_known']   = n_known.astype('int8')
    df['d_lower_bound'] = partial          # D >= partial holds on 100% of rows
    df['d_slack']       = D - partial      # NaN when D missing; = other + unknown parts

    # Impute daily: constraint first, weekend relationship as fallback (r = 0.80)
    d_imp = D.fillna(partial + n_known.map(SLACK_MED)).fillna(WK / WK_RATIO)
    df['daily_imp'] = d_imp
    df['daily_was_imputed'] = D.isna().astype('int8')

    # Each component is bounded above by whatever room the total leaves it
    for c, name in ((S_, 'social'), (G_, 'gaming'), (W_, 'work')):
        others = [x for x in (S_, G_, W_) if x != c]
        room = d_imp - df[others].fillna(0).sum(axis=1)
        df[f'{name}_room'] = room
        df[f'{name}_imp']  = df[c].fillna(room.clip(lower=0) * OTHER_FRAC[c])

    # Rebuild the most important derived features on the imputed values
    df['other_screen_imp']       = df['daily_imp'] - df['social_imp'] - df['gaming_imp'] - df['work_imp']
    df['screen_share_awake_imp'] = df['daily_imp'] / (24 - SL)
    df['weekend_imp']            = WK.fillna(df['daily_imp'] * WK_RATIO)
    df['week_avg_screen_imp']    = (5 * df['daily_imp'] + 2 * df['weekend_imp']) / 7
    return df


IMP_COLS = [
    'sgw_n_known', 'd_lower_bound', 'd_slack', 'daily_imp', 'daily_was_imputed',
    'social_room', 'gaming_room', 'work_room', 'social_imp', 'gaming_imp', 'work_imp',
    'other_screen_imp', 'screen_share_awake_imp', 'weekend_imp', 'week_avg_screen_imp',
]

train_fe = add_constraint_features(train_fe)
test_fe  = add_constraint_features(test_fe)

print(f'\n{len(IMP_COLS)} constraint features added')
print('daily still missing after imputation:',
      f"train {train_fe['daily_imp'].isna().mean():.2%} | test {test_fe['daily_imp'].isna().mean():.2%}")

# Does the imputed column actually carry the signal the raw one has?
_m = train_fe['daily_imp'].notna()
print(f"\nsingle-feature AUC  raw daily = "
      f"{roc_auc_score(train_fe.loc[train_fe[D_].notna(), TARGET], train_fe[D_].dropna()):.4f}"
      f"  |  imputed daily = {roc_auc_score(train_fe.loc[_m, TARGET], train_fe.loc[_m, 'daily_imp']):.4f}")

### Exact-value target encoding

The EDA above judged `notifications_per_day` (single-feature AUC 0.493) and `app_opens_per_day` (0.541)
to be at chance level. That reading was wrong, and the decile plot already contained the evidence:

```
app_opens_per_day, target rate by decile:
0.655, 0.734, 0.673, 0.645, 0.726, 0.716, 0.657, 0.763, 0.770, 0.755
```

Each decile holds ~61,000 rows, so the standard error of a rate is about 0.0019. The swing from 0.645 to
0.770 is roughly **65 standard errors** — not noise, but a real relationship that simply is not *monotonic*.
Single-feature AUC only measures monotonic separation, so a zig-zag relationship scores 0.50 while still
carrying substantial signal. The feature importance plot confirmed it independently: both columns ranked
4th and 5th by gain.

Exact-value target encoding is the tool that extracts this properly. Each numerical column is treated as a
categorical key and replaced by P(y = 1 | that exact value), estimated out-of-fold. Why it works here:

- Values repeat heavily — roughly 475 rows per distinct value — so each estimate rests on real support.
- Trees approximate P(y|x) with axis-aligned splits under `min_child_samples` regularisation, which smooths
  over exactly the kind of local structure this encoding captures directly.
- Synthetic competition data tends to reproduce values from its source dataset, so exact-value matching
  picks up the generator's fingerprint rather than just a smooth trend.

Two safeguards: the encoding is computed **out-of-fold** (a fold's values never see their own targets), and
rare values are shrunk toward the global mean with additive smoothing. The test set is encoded with a table
fitted on the full training set.

In [ ]:
# --- Out-of-fold target encoding ---
# The fold split is created here because the encoding must be built fold-wise, and every model
# below reuses exactly these folds so the OOF encoding is never contaminated.
y_full = train_fe[TARGET].values
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train_fe, y_full))
GLOBAL_MEAN = y_full.mean()

# Smoothing pulls rare values toward the global mean: (sum + k*prior) / (count + k).
# With ~475 rows per distinct value the raw mean is already stable, so k stays small.
TE_SMOOTH = 20

# Round 3 measured every encoding against its raw column. For the three smooth, strongly monotonic
# features the encoding scored *worse* as a standalone predictor (daily -0.013, weekend -0.016,
# social -0.034): there is no zig-zag to recover, so the per-value estimate is just a noisier
# version of the raw value. Skipping them is worth testing — but note the evidence says the TE
# column is a weaker *single* predictor, not that it harms a model holding both, so expect a small
# effect either way. Set TE_SKIP = [] to reproduce round 3.
TE_SKIP = ['daily_screen_time_hours', 'weekend_screen_time', 'social_media_hours']

# How much support does each distinct value actually have? This is what makes exact-value
# encoding viable rather than pure noise-fitting.
support = pd.DataFrame({
    'n_unique': [train_fe[c].nunique() for c in NUM_COLS],
    'rows_per_value': [round(train_fe[c].notna().sum() / max(train_fe[c].nunique(), 1)) for c in NUM_COLS],
}, index=NUM_COLS).sort_values('rows_per_value')
display(support)


def oof_target_encode(key_tr, key_te, smooth):
    """Encode a categorical key by P(y=1 | key), out-of-fold for train, full-train table for test."""
    def table(idx):
        g = (pd.DataFrame({'k': key_tr.iloc[idx].to_numpy(), 'y': y_full[idx]})
             .groupby('k')['y'].agg(['sum', 'count']))
        return (g['sum'] + smooth * GLOBAL_MEAN) / (g['count'] + smooth)

    oof = np.full(len(key_tr), np.nan)
    for tr_idx, va_idx in FOLDS:
        oof[va_idx] = key_tr.iloc[va_idx].map(table(tr_idx)).to_numpy(dtype='float64')
    oof = np.where(np.isnan(oof), GLOBAL_MEAN, oof)

    full = table(np.arange(len(key_tr)))
    te = key_te.map(full).astype('float64').fillna(GLOBAL_MEAN).to_numpy()
    return oof, te


TE_COLS = []
for col in NUM_COLS:
    if col in TE_SKIP:
        continue
    enc = f'{col}_te'
    train_fe[enc], test_fe[enc] = oof_target_encode(
        train_fe[col].astype('string').fillna('__NA__'),
        test_fe[col].astype('string').fillna('__NA__'),
        TE_SMOOTH,
    )
    TE_COLS.append(enc)

FEATURES = NUM_COLS + CAT_COLS + ENG_COLS + IMP_COLS + TE_COLS
print(f'\n{len(TE_COLS)} single-column encodings (skipped {len(TE_SKIP)}) -> {len(FEATURES)} features')

# Each encoded column on its own is a 1-D estimate of P(y | exact value). Comparing its AUC to
# the raw column shows how much non-monotonic structure the raw feature was hiding.
_encoded = [c for c in NUM_COLS if c not in TE_SKIP]
te_gain = pd.DataFrame({
    'raw_auc': [roc_auc_score(y_full[train_fe[c].notna()], train_fe[c].dropna()) for c in _encoded],
    'te_auc': [roc_auc_score(y_full, train_fe[f'{c}_te']) for c in _encoded],
}, index=_encoded)
te_gain['delta'] = (te_gain['te_auc'] - te_gain['raw_auc']).round(4)
display(te_gain.round(4).sort_values('delta', ascending=False))

In [ ]:
# --- Pairwise target encoding ---
# notifications_per_day and app_opens_per_day produced by far the strongest single encodings
# (+0.256 and +0.194 AUC over their raw columns), so their interaction is the obvious next place
# to look. Encoding raw value pairs would leave ~18 rows per combination — far too sparse to
# estimate a rate from — so each column is quantile-binned first and the smoothing is raised.
PAIR_BINS = 25
PAIR_SMOOTH = 50

TE_PAIRS = [
    ('notifications_per_day', 'app_opens_per_day'),
    ('notifications_per_day', 'sleep_hours'),
    ('app_opens_per_day', 'age'),
]

_bin_cache = {}


def binned_key(col):
    """Quantile-bin a column using edges fitted on train, applied identically to test."""
    if col not in _bin_cache:
        edges = pd.qcut(train_fe[col], PAIR_BINS, duplicates='drop', retbins=True)[1].copy()
        edges[0], edges[-1] = -np.inf, np.inf
        _bin_cache[col] = (
            pd.cut(train_fe[col], edges, labels=False).astype('string').fillna('__NA__'),
            pd.cut(test_fe[col], edges, labels=False).astype('string').fillna('__NA__'),
        )
    return _bin_cache[col]


PAIR_COLS = []
for a, b in TE_PAIRS:
    ka_tr, ka_te = binned_key(a)
    kb_tr, kb_te = binned_key(b)
    key_tr, key_te = ka_tr + '|' + kb_tr, ka_te + '|' + kb_te

    enc = f'{a}_X_{b}_te'
    train_fe[enc], test_fe[enc] = oof_target_encode(key_tr, key_te, PAIR_SMOOTH)
    PAIR_COLS.append(enc)

    cells = key_tr.nunique()
    print(f'{enc:52s} AUC={roc_auc_score(y_full, train_fe[enc]):.4f}  '
          f'{cells:5d} cells  ~{len(train_fe) // max(cells, 1):4d} rows/cell')

FEATURES = FEATURES + PAIR_COLS
print(f'\n{len(PAIR_COLS)} pair encodings -> {len(FEATURES)} features total')

In [ ]:
# Per-model matrices
X = train_fe[FEATURES].copy()
y = train_fe[TARGET].values
X_test = test_fe[FEATURES].copy()

# LightGBM / XGBoost: pandas 'category' dtype preserves NaN naturally
for c in CAT_COLS:
    cats = sorted(set(X[c].dropna()) | set(X_test[c].dropna()))
    X[c] = pd.Categorical(X[c], categories=cats)
    X_test[c] = pd.Categorical(X_test[c], categories=cats)

# CatBoost rejects NaN in categorical columns -> encode missingness as its own level
X_cb = X.copy()
X_test_cb = X_test.copy()
for c in CAT_COLS:
    X_cb[c] = X_cb[c].astype(object).where(X_cb[c].notna(), 'Missing').astype(str)
    X_test_cb[c] = X_test_cb[c].astype(object).where(X_test_cb[c].notna(), 'Missing').astype(str)
CAT_IDX = [FEATURES.index(c) for c in CAT_COLS]

print('X:', X.shape, '| X_test:', X_test.shape)
print('positive rate:', y.mean().round(4))

## 4. Modeling

10-fold `StratifiedKFold` with three gradient boosting models, then a weight optimisation over the OOF
predictions to blend them. Every model reuses the folds created in section 3 — the same split the target
encoding was built on, which is required for the OOF encoding to stay honest.

We report both **AUC** and **log loss** per model. AUC is the competition metric; log loss is kept as a
secondary read on calibration quality.

In [ ]:
# FOLDS was created in section 3 alongside the target encoding. Every model reuses those exact
# folds — using a different split here would let each model validate on rows whose encoding was
# fitted with their own targets, silently inflating the OOF score.
assert len(FOLDS) == N_SPLITS, 'run the target encoding cell first'

oof_preds  = {}
test_preds = {}


def report(name, oof):
    auc = roc_auc_score(y, oof)
    ll  = log_loss(y, oof)
    print(f'\n{name}  OOF AUC = {auc:.6f} | log loss = {ll:.6f}')
    return auc, ll

In [ ]:
%%time
# --- LightGBM ---
# In round 1 the mean best_iteration was 2466/3000, so LightGBM was the only model that actually
# reached early stopping. The ceiling is still raised to 5000 so it is never the limiting factor.
lgb_params = dict(
    objective='binary', metric='auc', n_estimators=5000, learning_rate=0.03,
    num_leaves=64, min_child_samples=60, colsample_bytree=0.8,
    subsample=0.8, subsample_freq=1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

oof = np.zeros(len(X)); pred = np.zeros(len(X_test)); best_iters = []

for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(200, verbose=False)],
    )
    oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
    best_iters.append(model.best_iteration_)
    print(f'fold {fold}  auc={roc_auc_score(y[va_idx], oof[va_idx]):.6f}  iter={model.best_iteration_}')

oof_preds['lgb'], test_preds['lgb'] = oof, pred
lgb_model = model
report('LightGBM', oof)
print('mean best_iteration:', int(np.mean(best_iters)))

In [ ]:
%%time
# --- XGBoost ---
# n_estimators raised 3000 -> 8000: in round 1 the folds hit the ceiling at 2973/2999, meaning none
# of them reached early stopping and the model was left undertrained.
xgb_params = dict(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    n_estimators=8000, learning_rate=0.03, max_depth=7,
    min_child_weight=20, colsample_bytree=0.8, subsample=0.8,
    reg_lambda=1.0, enable_categorical=True, max_cat_to_onehot=8,
    random_state=SEED, n_jobs=-1, early_stopping_rounds=200,
)

oof = np.zeros(len(X)); pred = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X.iloc[tr_idx], y[tr_idx],
              eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=False)
    oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'fold {fold}  auc={roc_auc_score(y[va_idx], oof[va_idx]):.6f}  iter={model.best_iteration}')

oof_preds['xgb'], test_preds['xgb'] = oof, pred
report('XGBoost', oof)

In [ ]:
%%time
# --- CatBoost ---
# Optional. At 10 folds this roughly doubles total runtime, and CatBoost has been the weakest of
# the three while correlating 0.993+ with the others. Set to False to skip it.
RUN_CATBOOST = True

# 1) eval_metric='AUC' makes CatBoost compute validation AUC at every iteration, which on its own
#    slowed training by roughly 10x. Early stopping runs on Logloss instead; AUC is still
#    reported per fold below.
# 2) iterations raised 3000 -> 8000: in round 1 all five folds hit the ceiling at 2996-2999.
cb_params = dict(
    loss_function='Logloss', eval_metric='Logloss', iterations=8000,
    learning_rate=0.05, depth=7, l2_leaf_reg=3.0,
    random_seed=SEED, verbose=0, allow_writing_files=False,
)

if RUN_CATBOOST:
    oof = np.zeros(len(X)); pred = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        model = cb.CatBoostClassifier(**cb_params)
        model.fit(
            X_cb.iloc[tr_idx], y[tr_idx],
            eval_set=(X_cb.iloc[va_idx], y[va_idx]),
            cat_features=CAT_IDX, early_stopping_rounds=200, verbose=0,
        )
        oof[va_idx] = model.predict_proba(X_cb.iloc[va_idx])[:, 1]
        pred += model.predict_proba(X_test_cb)[:, 1] / N_SPLITS
        print(f'fold {fold}  auc={roc_auc_score(y[va_idx], oof[va_idx]):.6f}  iter={model.get_best_iteration()}')

    oof_preds['cat'], test_preds['cat'] = oof, pred
    report('CatBoost', oof)
else:
    print('CatBoost skipped')

In [ ]:
# Model comparison
summary = pd.DataFrame({
    name: {'AUC': roc_auc_score(y, p), 'LogLoss': log_loss(y, p)}
    for name, p in oof_preds.items()
}).T.round(6)
display(summary)

# Correlation between OOF predictions -> tells us how much a blend can possibly help
oof_mat = pd.DataFrame(oof_preds)
print('\nOOF prediction correlation:')
display(oof_mat.corr().round(4))

In [ ]:
# Blend: probability averaging vs rank averaging
# The metric is AUC, so only the RANKING matters, not calibration. The three models produce
# probabilities on different scales, which means naive probability averaging can distort the
# ranking; rank averaging removes that problem entirely. We measure both and keep the winner.
# Single models are included as candidates too — round 1 showed the blend does not reliably beat
# the best single model, so "blend is always better" is not assumed here.
names = list(oof_preds.keys())
P_oof  = np.column_stack([oof_preds[n] for n in names])
P_test = np.column_stack([test_preds[n] for n in names])

R_oof  = np.column_stack([rankdata(oof_preds[n]) / len(y) for n in names])
R_test = np.column_stack([rankdata(test_preds[n]) / len(X_test) for n in names])


def optimize_weights(M):
    def loss(w):
        w = np.abs(w); w = w / w.sum()
        p = np.clip(M @ w, 1e-7, 1 - 1e-7)
        return -roc_auc_score(y, p) if METRIC == 'auc' else log_loss(y, p)
    res = minimize(loss, x0=np.ones(M.shape[1]) / M.shape[1], method='Nelder-Mead',
                   options={'maxiter': 300, 'xatol': 1e-4, 'fatol': 1e-7})
    w = np.abs(res.x)
    return w / w.sum()


w_prob = optimize_weights(P_oof)
w_rank = optimize_weights(R_oof)

candidates = {
    'prob_mean':      (P_oof.mean(1),   P_test.mean(1)),
    'prob_optimized': (P_oof @ w_prob,  P_test @ w_prob),
    'rank_mean':      (R_oof.mean(1),   R_test.mean(1)),
    'rank_optimized': (R_oof @ w_rank,  R_test @ w_rank),
}
for n in names:
    candidates[f'single_{n}'] = (oof_preds[n], test_preds[n])

scores = pd.Series({k: roc_auc_score(y, v[0]) for k, v in candidates.items()}).sort_values(ascending=False)
print('OOF AUC:'); print(scores.round(6).to_string())
print('\nprobability weights:', dict(zip(names, w_prob.round(4))))
print('rank weights       :', dict(zip(names, w_rank.round(4))))

BEST = scores.index[0]
blend_oof, blend_test = candidates[BEST]
print(f'\nselected -> {BEST}  (OOF AUC {scores[BEST]:.6f})')

# Note: rank-based outputs live in [0, 1] but are NOT probabilities. That is fine for AUC;
# it would be invalid if the metric were log loss.
if BEST.startswith('rank'):
    print('NOTE: submission contains ranks, not probabilities (valid for AUC).')

In [ ]:
# Feature importance (LightGBM, gain)
imp = pd.DataFrame({
    'feature': FEATURES,
    'gain': lgb_model.booster_.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=False)

fig, ax = plt.subplots(figsize=(9, 10))
sns.barplot(data=imp, y='feature', x='gain', palette='Blues_r', ax=ax)
ax.set_title('LightGBM Feature Importance (gain)')
plt.tight_layout(); plt.show()

display(imp.reset_index(drop=True))

In [ ]:
# Submission
# A single file: whichever candidate the blend cell selected by OOF AUC (BEST).
# In round 1 the blend and the best single model were indistinguishable on the LB
# (0.96632 vs 0.96636, a 0.00004 gap = noise) with a Spearman correlation of 0.9993,
# so writing and uploading the individual model files just burns submission budget.
import os

os.makedirs('../submissions', exist_ok=True)

submission = pd.DataFrame({
    'id': test_df['id'].values,
    TARGET: blend_test,
})
submission.to_csv('../submissions/submission.csv', index=False)

print('selected candidate:', BEST)
print('written           : ../submissions/submission.csv', submission.shape)
if BEST.startswith('rank'):
    print('output is on a rank scale (0-1), not probabilities -> valid for AUC')
else:
    print('pred mean:', submission[TARGET].mean().round(4),
          '| train positive rate:', y.mean().round(4))

# To write out an individual model as well (test_preds is still in memory):
# pd.DataFrame({'id': test_df['id'].values, TARGET: test_preds['xgb']}) \
#   .to_csv('../submissions/submission_xgb.csv', index=False)

submission.head()

In [ ]:
# Sanity check on the prediction distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(blend_oof, bins=60, ax=axes[0], color='#2b6cb0')
axes[0].set_title('OOF prediction distribution')
sns.histplot(blend_test, bins=60, ax=axes[1], color='#38a169')
axes[1].set_title('Test prediction distribution')
plt.tight_layout(); plt.show()

## 5. Results and Next Steps

### Leaderboard progress

| Round | Change | OOF AUC | LB |
|---|---|---|---|
| 1 | Baseline, 3000-iteration ceiling — XGBoost (5-fold) | 0.964864 | 0.96636 |
| 1 | Baseline blend, 0.69 xgb / 0.18 lgb / 0.13 cat | 0.964974 | 0.96632 |
| 2 | Raised iteration ceilings + rank blending option | — | 0.96647 |
| 3 | Constraint imputation + exact-value TE + 10-fold | 0.968456 | **0.96963** |
| 4 | Selective TE (skip smooth columns) + pairwise TE | — | — |

Round 3 gained **+0.00316** on the leaderboard, more than every earlier round combined.

Round 3 OOF detail (10-fold): LightGBM 0.968094, XGBoost 0.968265, CatBoost 0.968071,
blend 0.968456 at weights 0.22 / 0.44 / 0.34. CatBoost went from clearly weakest in round 1
(0.963607, weight 0.13) to level with the others. All four blend variants — probability/rank ×
mean/optimised — landed within 0.00001 of each other, and the blend beat the best single model by
only 0.00019, so the ensemble is still contributing very little.

**The CV is trustworthy.** OOF 0.968456 → LB 0.96963 is a +0.00117 offset, against ~+0.0014 in rounds
1–2. The fold-sharing between the target encoding and the CV was a real concern, but if it were inflating
OOF meaningfully the leaderboard would have fallen well short; instead it landed ~0.0002 below the
projection, inside noise. Nested encoding is therefore not urgent.

### What round 3 proved

The earlier EDA labelled `notifications_per_day` (AUC 0.492) and `app_opens_per_day` (0.541) as
chance-level features. Exact-value target encoding shows how wrong that was:

| Feature | raw AUC | TE AUC | delta |
|---|---|---|---|
| `notifications_per_day` | 0.4921 | **0.7485** | **+0.2564** |
| `app_opens_per_day` | 0.5409 | **0.7349** | **+0.1940** |
| `sleep_hours` | 0.5270 | 0.5914 | +0.0645 |
| `age` | 0.5023 | 0.5500 | +0.0477 |
| `work_study_hours` | 0.6549 | 0.6676 | +0.0127 |
| `gaming_hours` | 0.6220 | 0.6325 | +0.0105 |
| `daily_screen_time_hours` | 0.8896 | 0.8769 | −0.0126 |
| `weekend_screen_time` | 0.8810 | 0.8646 | −0.0164 |
| `social_media_hours` | 0.8578 | 0.8237 | −0.0341 |

`notifications_per_day` carries nearly as much signal as `social_media_hours` — it was simply invisible to
any monotonic summary statistic. **Single-feature AUC measures monotonic separation, not information.**
A zig-zag relationship scores 0.50 while being highly predictive, and the decile plot in section 2 had
shown exactly that pattern all along.

The bottom three rows are the mirror image of the same principle: where a relationship really is smooth and
monotonic, per-value encoding *loses* accuracy because it replaces a clean signal with a noisier estimate.
Round 4 skips encoding for those three via `TE_SKIP`.

Constraint imputation coverage came out at 4.40% of train rows (daily missing, all parts known) against a
7% guess, plus 25% of rows where daily is known and a part is missing — the case the `*_room` upper bounds
address. Both changes shipped together, so their individual contributions are not separable from these
numbers; given the table above, the encoding is very likely the dominant term.

**Generalisable takeaway:** on synthetic tabular competitions, check whether numerical columns behave like
high-cardinality categoricals before treating them as continuous, and look for arithmetic identities
between columns before accepting native NaN handling. Read the binned response curve, not the correlation.

### Candidate next steps

1. **Ablation:** drop `IMP_COLS` and rerun one LightGBM to separate the imputation contribution from TE.
   Worth knowing before investing further in either direction.
2. **Tune `TE_SMOOTH`, `PAIR_BINS`, `PAIR_SMOOTH`.** The current values are reasoned starting points, not
   optimised ones.
3. **More pairs**, if the three in round 4 pay off. `sleep_hours × age` and pairs involving the
   engineered ratio columns are untested.
4. **Nested target encoding** — not urgent given the CV/LB agreement above, but it would remove the last
   structural doubt about the OOF number.
5. **Original dataset augmentation** and **Optuna tuning on XGBoost** remain open, both smaller.